# 🏛️ Unity Catalog — Governance Deep Dive
### Databricks Free Edition | Hands-On Demo

**What we will cover in this notebook:**

| Step | Topic |
|------|-------|
| 1 | Explore the default metastore & catalogs |
| 2 | Create a Catalog |
| 3 | Create Schemas (Medallion pattern) |
| 4 | Create Objects — Table, View, Volume |
| 5 | Create User Groups (Account Console walkthrough) |
| 6 | GRANT different privileges to groups |
| 7 | SHOW GRANTS — verify who has what access |
| 8 | REVOKE privileges |
| 9 | Row-Level & Column-Level Security via Views |
| 10 | Clean-up |

> 💡 **Tip:** Run each cell one at a time and observe the output. Every `-- comment` explains *why*, not just *what*.

---
## 🔍 STEP 1 — Explore What Already Exists

Unity Catalog comes pre-configured in Databricks Free Edition.  
There is already a **metastore** attached to your workspace and a default catalog called `main`.  
Let's explore it first before creating anything new.

In [0]:
%sql
-- List all catalogs visible to you in this metastore
-- You should see at least: main, hive_metastore, samples
SHOW CATALOGS;

In [0]:
%sql
-- Look at the built-in 'samples' catalog that Databricks provides
USE CATALOG samples;
SHOW SCHEMAS;

In [0]:
%sql
-- Peek at the tables inside the tpch schema (sample data)
SHOW TABLES IN samples.tpch;

In [0]:
%sql
-- Quick data preview from a sample table
SELECT * FROM samples.tpch.customer LIMIT 5;

---
## 🗂️ STEP 2 — Create Our Own Catalog

### What is a Catalog?
A **catalog** is the top-level container in the 3-level namespace:

```
catalog  →  schema  →  table / view / volume / function
```

Think of it like a **database server** — everything lives inside it.

> ⚠️ **Free Edition note:** You need the `CREATE CATALOG` privilege on the metastore.  
> In Free Edition, your account admin role grants this by default.

In [0]:
%sql
-- Create our training catalog
-- IF NOT EXISTS means: safe to re-run, won't error if it already exists
CREATE CATALOG IF NOT EXISTS training_catalog
  COMMENT 'Governance training demo catalog — created during UC hands-on session';

In [0]:
%sql
-- Verify it was created
SHOW CATALOGS LIKE 'training*';

In [0]:
%sql
-- Describe the catalog to see its properties
DESCRIBE CATALOG EXTENDED training_catalog;

In [0]:
%sql
-- Set training_catalog as the default for this session
-- All subsequent SQL will resolve objects inside this catalog by default
USE CATALOG training_catalog;

---
## 📂 STEP 3 — Create Schemas (Medallion Architecture)

### What is a Schema?
A **schema** (also called a database) groups related objects inside a catalog.

We will follow the **Medallion Architecture** pattern — an industry standard in Databricks:

| Layer | Schema | Purpose |
|-------|--------|---------|
| 🥉 Bronze | `bronze` | Raw, unmodified ingested data |
| 🥈 Silver | `silver` | Cleaned, validated, deduplicated |
| 🥇 Gold   | `gold`   | Business aggregates, ready for analytics |

**Why does this matter for governance?**  
You can grant different groups different access per layer —  
e.g., only engineers can write to bronze, but analysts can only read from gold.

In [0]:
%sql
-- We are already using training_catalog (set in previous step)
-- Create all 3 medallion schemas

CREATE SCHEMA IF NOT EXISTS bronze
  COMMENT 'Raw ingested data — landing zone. No transformations applied.';

CREATE SCHEMA IF NOT EXISTS silver
  COMMENT 'Cleaned and validated data. Business rules applied.';

CREATE SCHEMA IF NOT EXISTS gold
  COMMENT 'Business aggregates and serving layer. Analytics-ready.';

In [0]:
%sql
-- Confirm all 3 schemas were created
SHOW SCHEMAS IN training_catalog;

In [0]:
%sql
-- Describe a schema to see its metadata
DESCRIBE SCHEMA EXTENDED training_catalog.bronze;

---
## 📋 STEP 4 — Create Objects

### What objects can live inside a Schema?

| Object | Description |
|--------|-------------|
| **Table** (Managed) | Delta table fully managed by Unity Catalog |
| **Table** (External) | Data stored in your own cloud storage |
| **View** | Virtual table — SQL query saved as a named object |
| **Volume** | Managed storage for unstructured files (CSV, JSON, images…) |
| **Function** | User-defined function (UDF) |

We will create examples of the first three.

### 4a. Managed Table in `bronze`

In [0]:
%sql
-- Create a Delta managed table in the bronze schema
-- Unity Catalog manages the storage location automatically
CREATE TABLE IF NOT EXISTS training_catalog.bronze.customers (
  customer_id   INT       COMMENT 'Unique customer identifier',
  full_name     STRING    COMMENT 'Customer full name',
  email         STRING    COMMENT 'Contact email — PII',
  city          STRING    COMMENT 'City of residence',
  region        STRING    COMMENT 'Sales region',
  signup_date   DATE      COMMENT 'Date customer registered'
)
USING DELTA
COMMENT 'Raw customer records loaded from CRM system';

In [0]:
%sql
-- Insert sample data so we have rows to query
INSERT INTO training_catalog.bronze.customers VALUES
  (1,  'Arjun Sharma',    'arjun@demo.com',   'Chennai',   'South', '2023-01-15'),
  (2,  'Priya Nair',      'priya@demo.com',   'Bangalore', 'South', '2023-02-20'),
  (3,  'Rahul Mehta',     'rahul@demo.com',   'Mumbai',    'West',  '2023-03-10'),
  (4,  'Sneha Reddy',     'sneha@demo.com',   'Hyderabad', 'South', '2023-04-05'),
  (5,  'Vikram Joshi',    'vikram@demo.com',  'Pune',      'West',  '2023-05-22'),
  (6,  'Deepa Iyer',      'deepa@demo.com',   'Chennai',   'South', '2023-06-18'),
  (7,  'Karthik Pillai',  'karthik@demo.com', 'Kochi',     'South', '2023-07-09'),
  (8,  'Ananya Das',      'ananya@demo.com',  'Kolkata',   'East',  '2023-08-14'),
  (9,  'Suresh Gupta',    'suresh@demo.com',  'Delhi',     'North', '2023-09-01'),
  (10, 'Meena Krishnan',  'meena@demo.com',   'Chennai',   'South', '2023-10-30');

In [0]:
%sql
-- Verify the data
SELECT * FROM training_catalog.bronze.customers;

In [0]:
%sql
-- Describe the table — shows column names, types, and comments
DESCRIBE TABLE EXTENDED training_catalog.bronze.customers;

### 4b. Create a Silver Table (cleaned data)

In [0]:
%sql
-- Silver layer: clean version — email masked, region standardized
CREATE TABLE IF NOT EXISTS training_catalog.silver.customers_clean
COMMENT 'Cleaned customer data — email masked, region validated'
AS
SELECT
  customer_id,
  full_name,
  CONCAT(LEFT(email, 2), '***@***.com')  AS email_masked,
  city,
  UPPER(region)                           AS region,
  signup_date,
  DATEDIFF(CURRENT_DATE(), signup_date)   AS days_as_customer
FROM training_catalog.bronze.customers;

In [0]:
%sql
SELECT * FROM training_catalog.silver.customers_clean;

### 4c. Create a View in `gold`

A **View** is not a copy of data — it is a saved SQL query.  
Governance tip: You can grant SELECT on a view without giving access to the underlying bronze table.

In [0]:
%sql
-- Gold view: city-level summary for business dashboards
CREATE OR REPLACE VIEW training_catalog.gold.city_summary
COMMENT 'Aggregated customer count and region per city — for BI dashboards'
AS
SELECT
  city,
  region,
  COUNT(*)         AS total_customers,
  MIN(signup_date) AS first_signup,
  MAX(signup_date) AS latest_signup
FROM training_catalog.bronze.customers
GROUP BY city, region
ORDER BY total_customers DESC;

In [0]:
%sql
SELECT * FROM training_catalog.gold.city_summary;

### 4d. Create a Volume in `bronze`

A **Volume** is Unity Catalog's managed object for **unstructured / semi-structured files**  
(CSV, JSON, Parquet, images, PDFs, etc.).  
It gives you a governed path: `/Volumes/catalog/schema/volume_name/`

In [0]:
%sql
-- Create a managed volume to store raw uploaded files
CREATE VOLUME IF NOT EXISTS training_catalog.bronze.raw_files
  COMMENT 'Landing zone for raw CSV and JSON files before ingestion';

In [0]:
%sql
-- List the volume path
LIST '/Volumes/training_catalog/bronze/raw_files';

In [0]:
# Write a sample file into the Volume using Python
sample_data = "customer_id,name,city\n101,Test User,Chennai\n102,Demo User,Mumbai\n"

dbutils.fs.put(
    "/Volumes/training_catalog/bronze/raw_files/sample_upload.csv",
    sample_data,
    overwrite=True
)
print("✅ File written to Volume successfully!")

In [0]:
%sql
-- Confirm the file is visible inside the Volume
LIST '/Volumes/training_catalog/bronze/raw_files';

---
## 👥 STEP 5 — Create User Groups

### ⚠️ Groups CANNOT be created via SQL in Unity Catalog.
Groups are managed at the **Account level** (not workspace level).

### How to create groups — UI walkthrough:

```
1. Go to:  https://accounts.azuredatabricks.net   (Azure)
           https://accounts.cloud.databricks.com  (AWS)

2. Click:  User Management  →  Groups  →  Add Group

3. Create these 3 groups:
   ┌─────────────────────┬──────────────────────────────────────────────┐
   │ Group Name          │ Represents                                   │
   ├─────────────────────┼──────────────────────────────────────────────┤
   │ data_engineers      │ Team that builds pipelines, writes to bronze │
   │ data_analysts       │ Team that reads gold layer for reports       │
   │ data_scientists     │ Team that needs specific table access        │
   └─────────────────────┴──────────────────────────────────────────────┘

4. Add members (users) to each group

5. Assign the groups to your workspace:
   Workspace Settings  →  Identity & Access  →  Add Group
```

> Once groups are created in Account Console, you can reference them in GRANT statements below.

> 💡 **For this demo:** If you don't have multiple users, you can GRANT to your own user email  
> or a service principal to see the effect.

---
## 🔐 STEP 6 — GRANT Permissions

### The GRANT chain — very important!

To access a table, a user/group needs privileges at **all 3 levels**:

```
USE CATALOG  →  USE SCHEMA  →  SELECT / MODIFY / etc.
```

Missing any one step = **access denied**. This is the most common mistake beginners make.

### 6a. Grant FULL ACCESS to `data_engineers`

Engineers need to create tables, write data, and manage the catalog.

In [0]:
%sql
-- ALL PRIVILEGES cascades down to all schemas and objects inside training_catalog
GRANT ALL PRIVILEGES
  ON CATALOG training_catalog
  TO `data_engineers`;

### 6b. Grant READ-ONLY access to `data_analysts` on the Gold layer

In [0]:
%sql
-- Step 1: Must grant USE CATALOG first
-- Without this, analysts cannot even see the catalog
GRANT USE CATALOG
  ON CATALOG training_catalog
  TO `data_analysts`;

In [0]:
%sql
-- Step 2: Grant USE SCHEMA + SELECT on gold schema only
-- Analysts should NOT see bronze or silver
GRANT USE SCHEMA, SELECT
  ON SCHEMA training_catalog.gold
  TO `data_analysts`;

### 6c. Grant access to `data_scientists` on a specific table only

In [0]:
%sql
-- Scientists need USE CATALOG first
GRANT USE CATALOG
  ON CATALOG training_catalog
  TO `data_scientists`;

In [0]:
%sql
-- Then USE SCHEMA on bronze
GRANT USE SCHEMA
  ON SCHEMA training_catalog.bronze
  TO `data_scientists`;

In [0]:
%sql
-- Finally, SELECT on one specific table only (not all tables in bronze)
GRANT SELECT
  ON TABLE training_catalog.bronze.customers
  TO `data_scientists`;

### 6d. Grant Volume access (READ + WRITE) to engineers

In [0]:
%sql
-- Engineers need to read AND write files to the volume
GRANT READ VOLUME, WRITE VOLUME
  ON VOLUME training_catalog.bronze.raw_files
  TO `data_engineers`;

### 6e. Grant CREATE TABLE on silver schema to engineers

In [0]:
%sql
-- Engineers should be able to create new tables and views in silver
GRANT CREATE TABLE, CREATE VIEW
  ON SCHEMA training_catalog.silver
  TO `data_engineers`;

---
## 🔎 STEP 7 — SHOW GRANTS (Verify Who Has What)

Always verify after granting. This is also how you audit access.

In [0]:
%sql
-- See all grants on the catalog level
SHOW GRANTS ON CATALOG training_catalog;

In [0]:
%sql
-- See grants on the gold schema
SHOW GRANTS ON SCHEMA training_catalog.gold;

In [0]:
%sql
-- See grants on the specific bronze customers table
SHOW GRANTS ON TABLE training_catalog.bronze.customers;

In [0]:
%sql
-- See grants on the volume
SHOW GRANTS ON VOLUME training_catalog.bronze.raw_files;

In [0]:
%sql
-- What can the data_analysts group do?
SHOW GRANTS TO `data_analysts`;

---
## 🚫 STEP 8 — REVOKE Permissions

REVOKE is the opposite of GRANT. Use it when:
- A team member moves to a different team
- A project ends and access should be removed
- You granted too much by mistake

In [0]:
%sql
-- Revoke SELECT from data_scientists on the customers table
REVOKE SELECT
  ON TABLE training_catalog.bronze.customers
  FROM `data_scientists`;

In [0]:
%sql
-- Confirm it was removed
SHOW GRANTS ON TABLE training_catalog.bronze.customers;

In [0]:
%sql
-- Revoke gold access from analysts
REVOKE USE SCHEMA, SELECT
  ON SCHEMA training_catalog.gold
  FROM `data_analysts`;

In [0]:
%sql
-- Verify gold schema has no analyst grant now
SHOW GRANTS ON SCHEMA training_catalog.gold;

---
## 🔒 STEP 9 — Row-Level & Column-Level Security

Unity Catalog implements row/column security simply using **Views** —  
grant access to the view, not the base table.  
This is called the **view-based security pattern**.

```
bronze.customers          ← engineers only (raw PII)
        ↓
gold.customers_masked     ← analysts (email hidden)
gold.customers_by_region  ← region teams (row filtered)
gold.customers_dynamic_mask ← dynamic masking per group
```

### 9a. Column-Level Security — Hide the email column

In [0]:
%sql
-- Create a view that masks the email column
-- Analysts query this view — they never touch the bronze table directly
CREATE OR REPLACE VIEW training_catalog.gold.customers_masked
COMMENT 'Customers view with email masked — safe for analyst access'
AS
SELECT
  customer_id,
  full_name,
  'REDACTED'   AS email,          -- column-level security: hide PII
  city,
  region,
  signup_date
FROM training_catalog.bronze.customers;

In [0]:
%sql
-- Grant analysts SELECT on the masked view (NOT on the bronze table)
GRANT USE SCHEMA, SELECT
  ON SCHEMA training_catalog.gold
  TO `data_analysts`;

GRANT SELECT
  ON VIEW training_catalog.gold.customers_masked
  TO `data_analysts`;

In [0]:
%sql
-- Analyst query: email is always REDACTED regardless of who runs it
SELECT * FROM training_catalog.gold.customers_masked;

### 9b. Row-Level Security — Each region sees only their own rows

In [0]:
%sql
-- Dynamic row filtering using IS_MEMBER()
-- IS_MEMBER('group_name') returns TRUE if the current user belongs to that group

CREATE OR REPLACE VIEW training_catalog.gold.customers_by_region
COMMENT 'Row-level security: South team sees only South, West team sees only West'
AS
SELECT *
FROM training_catalog.bronze.customers
WHERE
  (IS_MEMBER('south_team')    AND region = 'South')
  OR (IS_MEMBER('west_team')  AND region = 'West')
  OR (IS_MEMBER('data_engineers'));  -- engineers see all rows

In [0]:
%sql
-- Test: current user (admin/engineer) sees all rows
SELECT * FROM training_catalog.gold.customers_by_region;

### 9c. Dynamic Data Masking using IS_MEMBER()

In [0]:
%sql
-- Engineers see real email; everyone else sees a masked version
CREATE OR REPLACE VIEW training_catalog.gold.customers_dynamic_mask
COMMENT 'Email shown to engineers, masked for everyone else'
AS
SELECT
  customer_id,
  full_name,
  CASE
    WHEN IS_MEMBER('data_engineers') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.com')
  END AS email,
  city,
  region,
  signup_date
FROM training_catalog.bronze.customers;

In [0]:
%sql
-- Run as yourself (admin/engineer) — you see full emails
SELECT * FROM training_catalog.gold.customers_dynamic_mask;

---
## 📊 STEP 10 — Explore via Data Explorer (UI)

Everything we did in SQL is also visible and manageable in the **Databricks UI**.

### How to navigate:

```
Left sidebar  →  🗂️ Catalog icon

training_catalog
  └── bronze
        ├── customers           ← Schema tab, Sample Data tab, History tab
        └── raw_files (Volume)
  └── silver
        └── customers_clean
  └── gold
        ├── city_summary        ← Permissions tab → see grants
        ├── customers_masked
        ├── customers_by_region
        └── customers_dynamic_mask

On any object:
  → Permissions tab  : shows current grants, add/remove via UI (no SQL needed)
  → Details tab      : owner, created time, comments
  → Sample Data tab  : preview rows without writing SQL
  → History tab      : Delta table version history (time travel!)
```

---
## 🧹 STEP 11 — Clean Up

Run this section **only** when you are done with the demo and want to remove everything we created.

In [0]:
%sql
-- Revoke remaining grants before dropping (good practice)
REVOKE ALL PRIVILEGES ON CATALOG training_catalog FROM `data_engineers`;
REVOKE ALL PRIVILEGES ON CATALOG training_catalog FROM `data_analysts`;
REVOKE ALL PRIVILEGES ON CATALOG training_catalog FROM `data_scientists`;

In [0]:
%sql
-- Drop the entire catalog and everything inside it
-- CASCADE drops all schemas, tables, views, volumes inside
DROP CATALOG IF EXISTS training_catalog CASCADE;

In [0]:
%sql
-- Confirm it is gone
SHOW CATALOGS LIKE 'training*';

---
## 📝 Quick Reference Summary

### Privilege Cheat Sheet

| Privilege | Object | What it allows |
|-----------|--------|----------------|
| `ALL PRIVILEGES` | Catalog / Schema / Table | Full control |
| `USE CATALOG` | Catalog | Required to access anything inside |
| `USE SCHEMA` | Schema | Required to access objects inside |
| `SELECT` | Table / View | Read data |
| `MODIFY` | Table | INSERT, UPDATE, DELETE, MERGE |
| `CREATE TABLE` | Schema | Create new tables |
| `CREATE VIEW` | Schema | Create new views |
| `CREATE SCHEMA` | Catalog | Create new schemas |
| `CREATE VOLUME` | Schema | Create new volumes |
| `READ VOLUME` | Volume | Read files |
| `WRITE VOLUME` | Volume | Write files |
| `EXECUTE` | Function | Run a UDF |

### The Grant Chain (never forget this!)

```sql
-- To allow a group to SELECT from a table, you need ALL THREE:
GRANT USE CATALOG  ON CATALOG training_catalog            TO `my_group`;
GRANT USE SCHEMA   ON SCHEMA  training_catalog.bronze     TO `my_group`;
GRANT SELECT       ON TABLE   training_catalog.bronze.customers TO `my_group`;
```

### View-Based Security Pattern

```
bronze.customers              ← engineers only  (raw PII)
        ↓
gold.customers_masked         ← analysts        (email hidden)
gold.customers_by_region      ← region teams    (row filtered per group)
gold.customers_dynamic_mask   ← mixed access    (dynamic masking at query time)
```